# Colab runner

Runs `src/train.py` on a Colab GPU. Setup only; all model code lives in `src/`.

**Rule:** use a GPU runtime only for training. The machine's disk is wiped when the session ends; checkpoints are saved to Google Drive.

In [ ]:
# GPU check 

!nvidia-smi # the ! means run this as a terminal command on the Colab machine

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In later sessions, if the repo is already cloned, use !git pull instead

In [ ]:
# get the code 

!git clone https://github.com/yagmur-cam/mars-terrain-segmentation.git
%cd mars-terrain-segmentation
!git checkout private
!git pull
!grep num_workers src/train.py


#### Getting the data

In [ ]:
!df -h /content # check free disk space on Colab machine

In [ ]:
# copy the dataset from Drive to the fast local disk, then unzip

!cp /content/drive/MyDrive/ncam.7z /content/
!7z x -y -o/content/data /content/ncam.7z > /dev/null
!find /content/data -maxdepth 3 -type d

In [ ]:
# timing test, use a NEW --out_dir each time, or it resumes an old test

!python -m src.train --data_dir /content/data/ncam --out_dir /content/drive/MyDrive/mars_runs/test --size 256 --epochs 1 --limit 2000 --num_workers 2

In [ ]:
# start training
# baseline run (~2.5 h). If disconnected: rerun cells 2–5, then this exact command to resume

!python -m src.train --data_dir /content/data/ncam --out_dir /content/drive/MyDrive/mars_runs/run1_256 --size 256 --epochs 20 --num_workers 2

In [ ]:
# run2: class-weighted loss, everything else same as run1

!python -m src.train --data_dir /content/data/ncam --out_dir /content/drive/MyDrive/mars_runs/run2_weighted_256 --size 256 --epochs 20 --num_workers 2 --weighted

In [ ]:
# the test-set evaluation
# evaluate the best checkpoint on the expert test sets (min1, min2, min3)

!python -m src.evaluate --data_dir /content/data/ncam --checkpoint /content/drive/MyDrive/mars_runs/run1_256/best.pth --size 256 --out /content/drive/MyDrive/mars_runs/run1_256/results.json

In [ ]:
# test-set evaluation-2

!python -m src.evaluate --data_dir /content/data/ncam --checkpoint /content/drive/MyDrive/mars_runs/run2_weighted_256/best.pth --size 256 --out /content/drive/MyDrive/mars_runs/run2_weighted_256/results.json